# 패션 무드 모델 인터넷 이미지 테스트

`MODEL_NAME`만 변경하면 V1, V2, V2-2, V3, V4 체크포인트를 같은 이미지로 비교할 수 있습니다.

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms
import matplotlib.pyplot as plt

# 여기만 바꾸면 다른 모델을 테스트합니다.
MODEL_NAME = "v4"  # v1 | v2 | v2-2 | v3 | v4
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DATASET_PATH = Path("CNN_train_test/CNN_train_test_image")
TEST_DIR = Path("test_assets/internet_outfits/images")

CHECKPOINTS = {
    "v1": Path("Model/fashion_mood_model_pytorch.pth"),
    "v2": Path("Model/fashion_mood_model_pytorch2_epoch25_last3features.pth"),
    "v2-2": Path("Model/fashion_mood_model_pytorch2-2_more_layers_best.pth"),
    "v3": Path("Model/fashion_mood_model_pytorch3_street_cap_10000_best.pth"),
    "v4": Path("Model/fashion_mood_model_pytorch4_street_cap_10000_more_layers_best.pth"),
}

if MODEL_NAME not in CHECKPOINTS:
    raise ValueError(f"MODEL_NAME은 다음 중 하나여야 합니다: {list(CHECKPOINTS)}")
checkpoint_path = CHECKPOINTS[MODEL_NAME]
if not checkpoint_path.exists():
    raise FileNotFoundError(f"체크포인트가 없습니다: {checkpoint_path}")
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"데이터셋 경로가 없습니다: {DATASET_PATH}")

class_names = sorted(path.name for path in DATASET_PATH.iterdir() if path.is_dir())
num_classes = len(class_names)
print(f"선택 모델: {MODEL_NAME}")
print(f"체크포인트: {checkpoint_path}")
print(f"클래스 수: {num_classes}")
print(f"실행 장치: {DEVICE}")

In [ ]:
# 모든 체크포인트가 사용하는 ConvNeXt-Tiny 분류기 구성
mood_model = models.convnext_tiny(weights=None)
in_features = mood_model.classifier[2].in_features
mood_model.classifier = nn.Sequential(
    nn.Flatten(1),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes),
)
state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
mood_model.load_state_dict(state_dict)
mood_model = mood_model.to(DEVICE)
mood_model.eval()
print("모델 로드 완료")

In [ ]:
# 인터넷 이미지 10장 테스트
plt.rcParams["font.family"] = "Malgun Gothic"
image_paths = sorted(TEST_DIR.glob("*.jpg"))
if len(image_paths) != 10:
    raise RuntimeError(f"테스트 이미지가 {len(image_paths)}장입니다: {TEST_DIR}")

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
results = []
for image_path in image_paths:
    image = Image.open(image_path).convert("RGB")
    image_tensor = test_transform(image).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        probabilities = torch.softmax(mood_model(image_tensor), dim=1)[0]
    confidence, prediction = probabilities.max(dim=0)
    top_values, top_indices = probabilities.topk(3)
    true_name = image_path.stem.split("_", 1)[1]
    predicted_name = class_names[prediction.item()]
    results.append({
        "path": image_path,
        "image": image,
        "true": true_name,
        "predicted": predicted_name,
        "confidence": confidence.item(),
        "top3": [(class_names[i.item()], value.item()) for value, i in zip(top_values, top_indices)],
    })

correct_count = sum(result["true"] == result["predicted"] for result in results)
print(f"[{MODEL_NAME}] 인터넷 이미지 10장 중 {correct_count}장 정답 ({correct_count / 10:.1%})")
for result in results:
    mark = "O" if result["true"] == result["predicted"] else "X"
    top3_text = ", ".join(f"{name} {prob:.1%}" for name, prob in result["top3"])
    print(f"{mark} {result['path'].name} | 정답: {result['true']} | 예측: {result['predicted']} ({result['confidence']:.1%}) | Top-3: {top3_text}")

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for ax, result in zip(axes.flat, results):
    ax.imshow(result["image"])
    ax.axis("off")
    correct = result["true"] == result["predicted"]
    color = "green" if correct else "red"
    ax.set_title(f"정답: {result['true']}\n예측: {result['predicted']} ({result['confidence']:.1%})", color=color, fontsize=9)
plt.suptitle(f"{MODEL_NAME} 인터넷 이미지 테스트: {correct_count}/10")
plt.tight_layout()
plt.show()

In [ ]:
# 전체 검증 데이터 평가 및 혼동 행렬
# 인터넷 이미지 10장과 달리, 학습 때 사용한 동일한 80:20 검증 분할을 평가합니다.
import numpy as np
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

eval_dataset = datasets.ImageFolder(
    DATASET_PATH,
    transform=transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
)
targets = np.asarray(eval_dataset.targets)
all_indices = np.arange(len(eval_dataset))
_, val_indices = train_test_split(
    all_indices, test_size=0.2, stratify=targets, random_state=42,
)
val_dataset_for_test = Subset(eval_dataset, val_indices.tolist())
val_loader_for_test = DataLoader(val_dataset_for_test, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

actual_labels, predicted_labels = [], []
mood_model.eval()
with torch.inference_mode():
    for inputs, labels in val_loader_for_test:
        outputs = mood_model(inputs.to(DEVICE, non_blocking=True))
        predicted = outputs.argmax(dim=1).cpu().tolist()
        predicted_labels.extend(predicted)
        actual_labels.extend(labels.tolist())

matrix = confusion_matrix(actual_labels, predicted_labels, labels=list(range(num_classes)))
accuracy = accuracy_score(actual_labels, predicted_labels)
macro_f1 = f1_score(actual_labels, predicted_labels, labels=list(range(num_classes)), average="macro", zero_division=0)
balanced_acc = balanced_accuracy_score(actual_labels, predicted_labels)
print(f"[{MODEL_NAME}] Validation Accuracy: {accuracy:.2%}")
print(f"[{MODEL_NAME}] Validation Macro F1: {macro_f1:.4f}")
print(f"[{MODEL_NAME}] Validation Balanced Accuracy: {balanced_acc:.2%}")
print("\n클래스별 Precision / Recall / F1:")
print(classification_report(actual_labels, predicted_labels, target_names=class_names, labels=list(range(num_classes)), zero_division=0, digits=4))

fig, ax = plt.subplots(figsize=(16, 14))
display = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=class_names)
display.plot(ax=ax, cmap="Blues", xticks_rotation=90, colorbar=False, values_format="d")
ax.set_title(f"{MODEL_NAME} Validation Confusion Matrix")
ax.set_xlabel("예측 클래스")
ax.set_ylabel("실제 클래스")
plt.tight_layout()
plt.show()

In [ ]:
# 클래스별 전체 개수 대비 예측 비율 혼동행렬 (24 x 24)
# 각 행의 합계는 100%이며, 대각선은 클래스별 정답률(Recall)입니다.
matrix_np = np.asarray(matrix, dtype=float)
actual_count = matrix_np.sum(axis=1, keepdims=True)
normalized_matrix = np.divide(
    matrix_np,
    actual_count,
    out=np.zeros_like(matrix_np),
    where=actual_count != 0,
)
normalized_percent = normalized_matrix * 100

print(f"[{MODEL_NAME}] 클래스별 정답률")
for idx, class_name in enumerate(class_names):
    print(f"{class_name:<15}: {normalized_matrix[idx, idx]:.2%} ({int(matrix_np[idx, idx])}/{int(actual_count[idx, 0])})")

fig, ax = plt.subplots(figsize=(17, 15))
display = ConfusionMatrixDisplay(
    confusion_matrix=normalized_percent,
    display_labels=class_names,
)
display.plot(
    ax=ax,
    cmap="Blues",
    xticks_rotation=90,
    colorbar=True,
    values_format=".1f",
)
ax.set_title(f"{MODEL_NAME} 클래스별 비율 혼동행렬 (행 기준 %)")
ax.set_xlabel("예측 클래스")
ax.set_ylabel("실제 클래스")
plt.tight_layout()
plt.show()